
教学内容：封装完整的 RAG 问答流程（检索 + 生成）
核心功能：检索相关文档、构建 Prompt 上下文、调用 LLM 生成答案、
          返回答案 + 引用来源、多轮对话
前置知识：完成 rag_retrieval_api.py 的学习
###### 


In [ ]:
import os
import sys
from dotenv import load_dotenv


def demo_basic_qna():
    """演示基础 RAG 问答功能：初始化、添加文档、检索 + 生成"""
    print(f"\n-- 示例 1: 基础问答演示")

    # 初始化 RAG 问答系统
    qna = RAGQnA(
        milvus_uri=MILVUS_URI,
        collection_name="demo_qna",
        dim=DEFAULT_DIMENSION,
        llm_model="qwen-plus"
    )

    # 创建 Collection 并添加测试文档
    qna.retriever.create_collection()

    documents = [
        "机器学习是人工智能的核心技术，通过训练数据让计算机自动学习规律。机器学习需要数学基础，包括线性代数和概率统计。",
        "深度学习使用多层神经网络，在图像识别和自然语言处理领域取得成功。深度学习是机器学习的子集。",
        "自然语言处理让计算机理解和生成人类语言，应用包括机器翻译和智能客服。",
        "计算机视觉让计算机能够看懂图像，用于人脸识别和自动驾驶。",
        "Milvus 是一个开源的向量数据库，专门用于存储和搜索向量数据。",
    ]

    qna.retriever.add_documents(documents, chunk_size=100)

    # 问答测试
    questions = [
        "机器学习需要什么基础？",
        "深度学习有什么应用？"
    ]

    for question in questions:
        print(f"\n用户：{question}")

        result = qna.ask(question, top_k=3)

        print(f"助手：{result['answer']}")

        if result['sources']:
            print("\n引用来源:")
            for src in result['sources'][:2]:
                print(f"  [{src['id']}] {src['content']}")

    qna.close()


# =============================================================================
# 示例 2: Prompt 模板定制
# =============================================================================

In [ ]:
import os
import sys
from dotenv import load_dotenv


def demo_custom_prompt():
    """演示不同 Prompt 模板的构建方式和效果对比"""
    print(f"\n-- 示例 2: Prompt 模板定制")

    question = "机器学习是什么？"
    docs = [
        {"content": "机器学习是人工智能的核心技术。"},
        {"content": "机器学习通过数据训练模型。"},
    ]

    # 模板 1: 简洁版
    print("\n模板 1: 简洁版")
    print("-" * 50)
    context = "\n".join([f"[{i+1}] {d['content']}" for i, d in enumerate(docs)])
    prompt1 = f"""信息：{context}
问题：{question}
回答："""
    print(prompt1)

    # 模板 2: 详细版
    print("\n模板 2: 详细版（带要求）")
    print("-" * 50)
    prompt2 = f"""你是一个专业的 AI 助手。请根据以下信息回答问题。

相关信息：
{context}

问题：{question}

要求：
1. 基于上述信息回答
2. 引用来源用 [1]、[2] 标注
3. 回答简洁明了

回答："""
    print(prompt2)

    # 模板 3: 结构化版
    print("\n模板 3: 结构化版（分点回答）")
    print("-" * 50)
    prompt3 = f"""请根据以下信息回答问题：{question}

参考信息：
{context}

请按以下格式回答：
|【核心定义】用一句话概括
|【关键特点】列出 2-3 个特点
|【应用场景】列出 1-2 个应用场景

回答："""
    print(prompt3)


# =============================================================================
# 示例 3: 完整 RAG 流程解析
# =============================================================================

In [ ]:
import os
import sys
from dotenv import load_dotenv


def rag_pipeline_explained():
    """完整 RAG 流程解析（纯文档展示）"""
    print(f"\n-- 示例 3: 完整 RAG 流程解析")

    print("""
RAG 问答完整流程
--------------------------------------------------------------

  1. 用户提问
     → "机器学习需要什么基础？"

  2. 问题向量化
     → Embedding 模型编码 → [0.1, -0.5, 0.8, ...]

  3. 向量检索
     → 在 Milvus 中查找最相似的文档
     → 返回 Top-5 相关文档

  4. (可选) Rerank 重排序
     → 用 CrossEncoder 对结果重新打分
     → 提升相关性最高的文档排名

  5. 构建 Prompt
     → 拼接：系统指令 + 相关文档 + 用户问题

  6. LLM 生成答案
     → 基于 Prompt 生成回答
     → 支持流式输出

  7. 返回结果
     → 答案 + 引用来源 + 检索文档

关键设计点:

1. 检索质量决定上限
   - 如果检索不到相关文档，LLM 再强也没用
   - 建议：使用混合检索 + Rerank

2. Prompt 设计影响回答质量
   - 清晰的指令让 LLM 知道如何回答
   - 要求引用来源可以减少幻觉

3. 上下文长度限制
   - LLM 有 token 限制，不能传入太多文档
   - 建议：Top-K=5-10，或按 token 数控制

延迟分析:

  阶段       耗时        优化方法
  --------------------------------------------------
  检索       10-50ms     建立索引、批量查询
  Rerank     100-500ms   可跳过或用轻量模型
  LLM 生成   1-5s        流式输出、用小模型

关键概念总结:

  RAGQnA           封装检索 + 生成的完整流程
  ask()            问主治接口
  _build_prompt()  构建 RAG Prompt
  _generate()      调用 LLM 生成
  chat()           多轮对话
""")


# =============================================================================
# 主程序入口
# =============================================================================